<a href="https://colab.research.google.com/github/sw7779/Flood-Forecasting/blob/main/PDS_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modeling

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [ ]:
# Load dataset
df = pd.read_csv('https://drive.google.com/uc?id=1cBM91jrNnj6Ice3qr-rGWieRd8FfTAGg')

In [ ]:
# Display the first few rows
print("Dataset Overview:")
print(df.head())

Dataset Overview:
   MonsoonIntensity  TopographyDrainage  RiverManagement  Deforestation  \
0                 3                   8                6              6   
1                 8                   4                5              7   
2                 3                  10                4              1   
3                 4                   4                2              7   
4                 3                   7                5              2   

   Urbanization  ClimateChange  DamsQuality  Siltation  AgriculturalPractices  \
0             4              4            6          2                      3   
1             7              9            1          5                      5   
2             7              5            4          7                      4   
3             3              4            1          4                      6   
4             5              8            5          2                      7   

   Encroachments  ...  DrainageSystems  Coas

In [ ]:
# Feature Selection
X = df.drop('FloodProbability',axis=1)
y = df['FloodProbability']

# Scale the data
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=155)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=155)

In [ ]:
# Linear Regression
lr_model = LinearRegression()
lr_model.fit(x_train, y_train)

LinearRegression()

In [ ]:
# Random Forest Grid Search
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf_model = RandomForestRegressor(random_state=155)
rf_grid_search = GridSearchCV(estimator=rf_model, param_grid=rf_param_grid, cv=kf, scoring='r2', verbose=2)
rf_grid_search.fit(x_train, y_train)

# Best Parameters and Results
print("\nRandom Forest Results:")
print(f"Best Hyperparameters: {rf_grid_search.best_params_}")
print(f"Best Cross-Validation R² Score: {rf_grid_search.best_score_:.4f}")

Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  14.5s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  14.3s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  14.3s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  14.4s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  14.4s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  29.2s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  28.6s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  28.9s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  29.1s
[CV] END max_depth=10, min_sam

In [ ]:
# Wrapper class for compatibility
class SklearnCompatibleXGB(BaseEstimator, RegressorMixin):
    def __init__(self, **kwargs):
        self.model = XGBRegressor(**kwargs)
        self.params = kwargs  # Store parameters

    def fit(self, X, y):
        """Fit the model."""
        self.model.fit(X, y)
        return self

    def predict(self, X):
        """Predict using the model."""
        return self.model.predict(X)

    def get_params(self, deep=True):
        """Expose parameters for GridSearchCV."""
        return self.params

    def set_params(self, **params):
        """Set parameters for GridSearchCV."""
        self.model.set_params(**params)
        self.params.update(params)
        return self

In [ ]:
# XGBoost Grid Search
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9]
}

# Initialize XGBoost Model
xgb_model = SklearnCompatibleXGB(random_state=155)
xgb_grid_search = GridSearchCV(estimator=xgb_model, param_grid=xgb_param_grid,
                           cv=kf, scoring='r2', verbose=2)
xgb_grid_search.fit(x_train, y_train)

# Best Parameters and Results
print("\nXGBoost Results:")
print(f"Best Hyperparameters: {xgb_grid_search.best_params_}")
print(f"Best Cross-Validation R² Score: {xgb_grid_search.best_score_:.4f}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.7; total time=   0.4s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.7; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.7; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.7; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.7; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   0.3

In [ ]:
# LightGBM Grid Search
lgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

# Initialize LightGBM Model
lgb_model = LGBMRegressor(random_state=155)
lgb_grid_search = GridSearchCV(
    estimator=lgb_model,
    param_grid=lgb_param_grid,
    cv=kf,
    scoring='r2',
    verbose=2,
    n_jobs=-1  # Use all CPU cores for faster computation
)
lgb_grid_search.fit(x_train, y_train)

# Best Parameters and Results
print("\nLightGBM Results:")
print(f"Best Hyperparameters: {lgb_grid_search.best_params_}")
print(f"Best Cross-Validation R² Score: {lgb_grid_search.best_score_:.4f}")

Fitting 5 folds for each of 162 candidates, totalling 810 fits


/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMRegressor or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013605 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 342
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 20
[LightGBM] [Info] Start training from score 0.499885

LightGBM Results:
Best Hyperparameters: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.7}
Best Cross-Validation R² Score: 0.9681


In [ ]:
y_pred_lr = lr_model.predict(x_test)

r2_lr = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print(f'R-squared: {r2_lr}')
print(f'Mean Squared Error: {mse_lr}')
print(f"Root Mean Squared Error (RMSE): {rmse_lr}")

R-squared: 1.0
Mean Squared Error: 4.2335563391117224e-29
Root Mean Squared Error (RMSE): 6.506578470372676e-15


In [ ]:
y_pred_rf = rf_grid_search.predict(x_test)

r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)

print(f'R-squared: {r2_rf}')
print(f'Mean Squared Error: {mse_rf}')
print(f"Root Mean Squared Error (RMSE): {rmse_rf}")

R-squared: 0.7345302483264607
Mean Squared Error: 0.0006557942121745686
Root Mean Squared Error (RMSE): 0.025608479302265657


In [ ]:
y_pred_xgb = xgb_grid_search.predict(x_test)

r2_xgb = r2_score(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)

print(f'R-squared: {r2_xgb}')
print(f'Mean Squared Error: {mse_xgb}')
print(f"Root Mean Squared Error (RMSE): {rmse_xgb}")

R-squared: 0.9618458134826764
Mean Squared Error: 9.425290275277632e-05
Root Mean Squared Error (RMSE): 0.009708393417696685


In [ ]:
y_pred_lgb = lgb_grid_search.predict(x_test)

r2_lgb = r2_score(y_test, y_pred_lgb)
mse_lgb = mean_squared_error(y_test, y_pred_lgb)
rmse_lgb = np.sqrt(mse_lgb)

print(f'R-squared: {r2_lgb}')
print(f'Mean Squared Error: {mse_lgb}')
print(f"Root Mean Squared Error (RMSE): {rmse_lgb}")

R-squared: 0.9696925956289839
Mean Squared Error: 7.486887017165082e-05
Root Mean Squared Error (RMSE): 0.008652679941593288


/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMRegressor or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
results = []

results.append({
    "Model": "Linear Regression",
    "R²": r2_lr,
    "MSE": mse_lr,
    "RMSE": rmse_lr
})

results.append({
    "Model": "Random Forest",
    "R²": r2_rf,
    "MSE": mse_rf,
    "RMSE": rmse_rf
})

results.append({
    "Model": "XGBoost",
    "R²": r2_xgb,
    "MSE": mse_xgb,
    "RMSE": rmse_xgb
})

results.append({
    "Model": "LightGBM",
    "R²": r2_lgb,
    "MSE": mse_lgb,
    "RMSE": rmse_lgb
})

metrics_df = pd.DataFrame(results)
print(metrics_df)

               Model        R²           MSE          RMSE
0  Linear Regression  1.000000  4.233556e-29  6.506578e-15
1      Random Forest  0.734530  6.557942e-04  2.560848e-02
2            XGBoost  0.961846  9.425290e-05  9.708393e-03
3           LightGBM  0.969693  7.486887e-05  8.652680e-03
